In [1]:
import triton
print(triton.__version__)

3.6.0


In [ ]:
import torch
import triton
import triton.language as tl
import time

@triton.jit
def matmul_kernel(
    A_ptr, B_ptr, C_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,   # tile rows
    BLOCK_N: tl.constexpr,   # tile cols
    BLOCK_K: tl.constexpr,   # tile depth
):
    """
    Tiled matrix multiplication in Triton.
    Each program instance computes one BLOCK_M x BLOCK_N tile of C.
    Triton handles shared memory tiling, synchronization, and memory coalescing.
    """
    # Which tile are we computing?
    pid_m = tl.program_id(axis=0)   # tile row index
    pid_n = tl.program_id(axis=1)   # tile col index

    # Row and column offsets for this tile
    row_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # [BLOCK_M]
    col_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)  # [BLOCK_N]

    # Accumulator for this tile — lives in registers
    accumulator = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # Sweep across K dimension in BLOCK_K chunks (same as tiling in CUDA)
    for k_start in range(0, K, BLOCK_K):
        k_offsets = k_start + tl.arange(0, BLOCK_K)  # [BLOCK_K]

        # Load tile of A: shape [BLOCK_M, BLOCK_K]
        # Mask out-of-bounds accesses
        a_ptrs = A_ptr + row_offsets[:, None] * stride_am + k_offsets[None, :] * stride_ak
        a_mask = (row_offsets[:, None] < M) & (k_offsets[None, :] < K)
        a_tile = tl.load(a_ptrs, mask=a_mask, other=0.0)

        # Load tile of B: shape [BLOCK_K, BLOCK_N]
        b_ptrs = B_ptr + k_offsets[:, None] * stride_bk + col_offsets[None, :] * stride_bn
        b_mask = (k_offsets[:, None] < K) & (col_offsets[None, :] < N)
        b_tile = tl.load(b_ptrs, mask=b_mask, other=0.0)

        # Matrix multiply accumulate — Triton compiles this to Tensor Core ops
        accumulator += tl.dot(a_tile, b_tile)

    # Write output tile to C
    c_ptrs = C_ptr + row_offsets[:, None] * stride_cm + col_offsets[None, :] * stride_cn
    c_mask = (row_offsets[:, None] < M) & (col_offsets[None, :] < N)
    tl.store(c_ptrs, accumulator.to(tl.float16), mask=c_mask)


def triton_matmul(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    assert A.is_cuda and B.is_cuda

    C = torch.empty((M, N), dtype=torch.float16, device=A.device)

    # Block sizes — tuned for T4
    BLOCK_M, BLOCK_N, BLOCK_K = 128, 128, 32

    # Grid: one program instance per output tile
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))

    matmul_kernel[grid](
        A, B, C,
        M, N, K,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return C


# Correctness check — always verify before benchmarking
def verify_correctness(N: int = 512):
    A = torch.randn(N, N, dtype=torch.float16, device="cuda")
    B = torch.randn(N, N, dtype=torch.float16, device="cuda")

    C_triton = triton_matmul(A, B)
    C_ref = torch.matmul(A, B)

    max_diff = (C_triton - C_ref).abs().max().item()
    rel_err  = max_diff / C_ref.abs().max().item()
    print(f"Max absolute diff: {max_diff:.6f}")
    print(f"Relative error:    {rel_err:.6f}")
    assert rel_err < 0.01, f"Correctness check failed: {rel_err}"
    print("Correctness check passed.")

verify_correctness()

Max absolute diff: 0.062500
Relative error:    0.000529
Correctness check passed.


In [ ]:
# triton_matmul_autotuned.py
import torch
import triton
import triton.language as tl

# Define configs to search over
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64,  "BLOCK_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 128, "BLOCK_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 64,  "BLOCK_K": 64}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 64}, num_stages=4, num_warps=8),
        triton.Config({"BLOCK_M": 256, "BLOCK_N": 64,  "BLOCK_K": 32}, num_stages=4, num_warps=8),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 256, "BLOCK_K": 32}, num_stages=4, num_warps=8),
    ],
    key=["M", "N", "K"],   # retune when these change
)
@triton.jit
def matmul_kernel_autotuned(
    A_ptr, B_ptr, C_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)
    row_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    col_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    accumulator = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k_start in range(0, K, BLOCK_K):
        k_offsets = k_start + tl.arange(0, BLOCK_K)
        a_ptrs = A_ptr + row_offsets[:, None] * stride_am + k_offsets[None, :] * stride_ak
        b_ptrs = B_ptr + k_offsets[:, None] * stride_bk + col_offsets[None, :] * stride_bn
        a_tile = tl.load(a_ptrs, mask=(row_offsets[:, None] < M) & (k_offsets[None, :] < K), other=0.0)
        b_tile = tl.load(b_ptrs, mask=(k_offsets[:, None] < K) & (col_offsets[None, :] < N), other=0.0)
        accumulator += tl.dot(a_tile, b_tile)

    c_ptrs = C_ptr + row_offsets[:, None] * stride_cm + col_offsets[None, :] * stride_cn
    tl.store(c_ptrs, accumulator.to(tl.float16),
             mask=(row_offsets[:, None] < M) & (col_offsets[None, :] < N))


def triton_matmul_autotuned(A, B):
    M, K = A.shape
    _, N = B.shape
    C = torch.empty((M, N), dtype=torch.float16, device=A.device)
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_M"]), triton.cdiv(N, meta["BLOCK_N"]))
    matmul_kernel_autotuned[grid](
        A, B, C, M, N, K,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
    )
    return C